# Notebook 4: Graph Attention Networks (GAT)

GCN's fatal flaw: when aggregating, every neighbor gets the same weight (scaled only by degree). But clearly, not all neighbors are equally important. **GAT** fixes this by learning which neighbors to pay attention to.

**Prerequisites:** Notebooks 1–3

**What you'll learn:**
- The problem with uniform neighbor weighting in GCN
- How attention mechanisms work on graphs
- GAT math: single-head and multi-head attention
- Implementing a GAT attention layer from scratch
- Using PyG's `GATConv`
- Visualizing learned attention weights

**By the end:** implement 2-head GAT and visualize attention weights on a small graph.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.datasets import Planetoid, KarateClub
from torch_geometric.nn import GATConv, GCNConv
from torch_geometric.utils import to_networkx

torch.manual_seed(42)
np.random.seed(42)

dataset_cora = Planetoid(root='/tmp/Cora', name='Cora')
data_cora = dataset_cora[0]
dataset_karate = KarateClub()
data_karate = dataset_karate[0]

---
## 1. The Problem with GCN's Fixed Weights

In GCN, the aggregation for node $i$ is:
$$h_i^{(l+1)} = \sigma\left(\sum_{j \in \mathcal{N}(i) \cup \{i\}} \frac{1}{\sqrt{d_i d_j}} W h_j^{(l)}\right)$$

The weight $\frac{1}{\sqrt{d_i d_j}}$ depends only on **degree** — a structural property, not learned from the data. This means:
- A noisy/irrelevant neighbor has the same influence as a highly relevant one
- The model can't dynamically emphasize important connections

**GAT fix:** replace the fixed normalization with **learned, data-dependent attention coefficients**.

---
## 2. The GAT Attention Mechanism

**Paper:** Veličković et al., 2018 — "Graph Attention Networks"

### Step 1: Linear projection
For each node, project features: $\vec{h}_i' = W \vec{h}_i$

### Step 2: Compute attention logits (for each edge i→j)
$$e_{ij} = \text{LeakyReLU}(\vec{a}^T [\vec{h}_i' \| \vec{h}_j'])$$

Where $\vec{a}$ is a learnable vector, and $\|$ is concatenation.

### Step 3: Normalize over neighbors (softmax)
$$\alpha_{ij} = \text{softmax}_j(e_{ij}) = \frac{\exp(e_{ij})}{\sum_{k \in \mathcal{N}(i)} \exp(e_{ik})}$$

### Step 4: Weighted aggregation
$$\vec{h}_i^{\text{new}} = \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij} \vec{h}_j'\right)$$

### Multi-head attention
Run K independent attention mechanisms in parallel, then concatenate (or average) their outputs:
$$\vec{h}_i^{\text{new}} = \|_{k=1}^{K} \sigma\left(\sum_{j \in \mathcal{N}(i)} \alpha_{ij}^{(k)} W^{(k)} \vec{h}_j\right)$$

In [ ]:
# ============================================================
# GAT layer from scratch (single head)
# ============================================================
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax, add_self_loops

class GATLayerScratch(MessagePassing):
    """Single-head GAT layer, implementing the attention mechanism explicitly."""
    
    def __init__(self, in_channels: int, out_channels: int, negative_slope: float = 0.2):
        super().__init__(aggr='add')
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        # W: linear projection
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        # a: attention vector — operates on concatenated [h_i' || h_j']
        self.a = nn.Parameter(torch.empty(1, 2 * out_channels))
        self.leaky_relu = nn.LeakyReLU(negative_slope)
        
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a)
    
    def forward(self, x, edge_index):
        # Add self-loops
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        
        # Project all node features
        x_proj = self.W(x)  # (N, out_channels)
        
        return self.propagate(edge_index, x=x_proj)
    
    def message(self, x_i, x_j, index):
        # x_i: projected features of destination nodes [E, out_channels]
        # x_j: projected features of source nodes [E, out_channels]
        
        # Concatenate i and j features for attention
        concat = torch.cat([x_i, x_j], dim=-1)  # [E, 2*out_channels]
        e = self.leaky_relu((concat * self.a).sum(dim=-1))  # [E]
        
        # Normalize with softmax over each node's neighbors
        alpha = softmax(e, index)  # [E] — softmax per destination node
        
        # Weight source features by attention
        return alpha.unsqueeze(-1) * x_j  # [E, out_channels]


# Test
conv_scratch = GATLayerScratch(data_karate.num_features, 8)
out = conv_scratch(data_karate.x, data_karate.edge_index)
print(f'GAT scratch output shape: {out.shape}')
print('Single-head GAT layer works!')

In [ ]:
# ============================================================
# GAT with attention return — for visualization
# ============================================================
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax, add_self_loops

class GATLayerWithAttention(MessagePassing):
    """GAT layer that returns attention weights for visualization."""
    
    def __init__(self, in_channels, out_channels, negative_slope=0.2):
        super().__init__(aggr='add')
        self.W = nn.Linear(in_channels, out_channels, bias=False)
        self.a = nn.Parameter(torch.empty(1, 2 * out_channels))
        self.leaky_relu = nn.LeakyReLU(negative_slope)
        nn.init.xavier_uniform_(self.W.weight)
        nn.init.xavier_uniform_(self.a)
        self._alpha = None  # store attention weights
    
    def forward(self, x, edge_index):
        edge_index, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        x_proj = self.W(x)
        out = self.propagate(edge_index, x=x_proj)
        return out, self._alpha, edge_index
    
    def message(self, x_i, x_j, index):
        concat = torch.cat([x_i, x_j], dim=-1)
        e = self.leaky_relu((concat * self.a).sum(dim=-1))
        alpha = softmax(e, index)
        self._alpha = alpha.detach()  # save for inspection
        return alpha.unsqueeze(-1) * x_j

In [ ]:
# ============================================================
# Full GAT model using PyG's GATConv (production quality)
# ============================================================

class GAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 heads=8, dropout=0.6):
        super().__init__()
        self.dropout = dropout
        # First layer: 8 heads, concatenate outputs -> hidden * heads
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads,
                             dropout=dropout)
        # Last layer: 1 head, average outputs
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1,
                             concat=False, dropout=dropout)
    
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.elu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x
    
    def get_attention(self, x, edge_index):
        """Forward pass returning attention weights from both layers."""
        self.eval()
        with torch.no_grad():
            x_drop = F.dropout(x, p=0, training=False)
            out1, (ei1, alpha1) = self.conv1(x_drop, edge_index, return_attention_weights=True)
            x_elu = F.elu(out1)
            _, (ei2, alpha2) = self.conv2(x_elu, edge_index, return_attention_weights=True)
        return (ei1, alpha1), (ei2, alpha2)


torch.manual_seed(42)
model_gat = GAT(
    in_channels=dataset_cora.num_features,
    hidden_channels=8,      # per head
    out_channels=dataset_cora.num_classes,
    heads=8,
    dropout=0.6
)
optimizer_gat = torch.optim.Adam(model_gat.parameters(), lr=0.005, weight_decay=5e-4)

print(model_gat)
print(f'Total params: {sum(p.numel() for p in model_gat.parameters()):,}')

In [ ]:
def train_gat(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index)
    loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

@torch.no_grad()
def eval_gat(model, data):
    model.eval()
    out = model(data.x, data.edge_index)
    pred = out.argmax(1)
    accs = {}
    for split in ['train', 'val', 'test']:
        mask = getattr(data, f'{split}_mask')
        accs[split] = (pred[mask] == data.y[mask]).float().mean().item()
    return accs


history_gat = {'loss': [], 'val': [], 'test': []}
best_val = 0; best_test = 0

for epoch in range(1, 301):
    loss = train_gat(model_gat, data_cora, optimizer_gat)
    accs = eval_gat(model_gat, data_cora)
    history_gat['loss'].append(loss)
    history_gat['val'].append(accs['val'])
    history_gat['test'].append(accs['test'])
    if accs['val'] > best_val:
        best_val = accs['val']
        best_test = accs['test']
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Val: {accs["val"]:.4f} | Test: {accs["test"]:.4f}')

print(f'\nBest val: {best_val:.4f} | Test at best val: {best_test:.4f}')

---
## 3. Visualizing Attention Weights

The real power of GAT: we can *inspect* what the model learned to pay attention to. Let's visualize attention weights on the Karate Club graph.

In [ ]:
# Train a small GAT on Karate Club for visualization
torch.manual_seed(42)
model_karate = GAT(
    in_channels=data_karate.num_features,
    hidden_channels=4,
    out_channels=dataset_karate.num_classes,
    heads=2,
    dropout=0.3
)
optimizer_k = torch.optim.Adam(model_karate.parameters(), lr=0.01)

for epoch in range(300):
    model_karate.train()
    optimizer_k.zero_grad()
    out = model_karate(data_karate.x, data_karate.edge_index)
    loss = F.cross_entropy(out[data_karate.train_mask], data_karate.y[data_karate.train_mask])
    loss.backward(); optimizer_k.step()

# Get attention weights
(ei1, alpha1), (ei2, alpha2) = model_karate.get_attention(data_karate.x, data_karate.edge_index)
print(f'Layer 1 attention shape: {alpha1.shape}  (num_edges x num_heads)')
print(f'Layer 2 attention shape: {alpha2.shape}')

# Use head 0 of layer 1 for visualization
alpha_viz = alpha1[:, 0].numpy()  # (num_edges,)
print(f'Attention values: min={alpha_viz.min():.4f}, max={alpha_viz.max():.4f}, mean={alpha_viz.mean():.4f}')

In [ ]:
# Visualize attention weights as edge thickness/color
nx_g = to_networkx(data_karate, to_undirected=False)
node_colors = plt.cm.tab10(data_karate.y.numpy())

# Map edge_index edges to attention values
# ei1 includes self-loops, we want only real edges
num_edges_with_self = ei1.shape[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
pos = nx.spring_layout(nx_g.to_undirected(), seed=7)

# Left: uniform (GCN-like)
nx_g_undir = nx_g.to_undirected()
nx.draw(nx_g_undir, pos, ax=axes[0], node_color=node_colors, node_size=300,
        edge_color='gray', width=1.0, with_labels=True, font_size=7)
axes[0].set_title('GCN: uniform edge weights')

# Right: attention-weighted edges
# Get real (non-self-loop) edges and their attention
real_edges = [(ei1[0, i].item(), ei1[1, i].item())
              for i in range(num_edges_with_self)
              if ei1[0, i].item() != ei1[1, i].item()]
real_attns = [alpha_viz[i] for i in range(num_edges_with_self)
              if ei1[0, i].item() != ei1[1, i].item()]

# Normalize for visualization
max_attn = max(real_attns) if real_attns else 1
norm_attns = [a / max_attn for a in real_attns]

# Draw each edge with attention-based width
for (src, dst), attn in zip(real_edges, norm_attns):
    if src in nx_g_undir and dst in nx_g_undir[src]:
        nx.draw_networkx_edges(nx_g_undir, pos, edgelist=[(src, dst)],
                               width=attn * 6 + 0.3,
                               edge_color=[attn], edge_cmap=plt.cm.Reds,
                               alpha=0.7, ax=axes[1])
nx.draw_networkx_nodes(nx_g_undir, pos, node_color=node_colors, node_size=300, ax=axes[1])
nx.draw_networkx_labels(nx_g_undir, pos, font_size=7, ax=axes[1])
axes[1].set_title('GAT: edge thickness = attention weight')

plt.suptitle('GCN vs GAT — Karate Club', fontsize=13)
plt.tight_layout()
plt.show()

---
## 4. GCN vs. GAT: Head-to-Head on Cora

In [ ]:
# Quick GCN baseline for comparison
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.conv1 = GCNConv(in_channels, hidden_channels, cached=True)
        self.conv2 = GCNConv(hidden_channels, out_channels, cached=True)
    
    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def run_model(model, data, optimizer, epochs=300):
    best_val, best_test = 0, 0
    for epoch in range(epochs):
        model.train(); optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            pred = model(data.x, data.edge_index).argmax(1)
        val = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        test = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if val > best_val:
            best_val, best_test = val, test
    return best_val, best_test


torch.manual_seed(42)
gcn = GCN(dataset_cora.num_features, 64, dataset_cora.num_classes)
gcn_val, gcn_test = run_model(gcn, data_cora,
    torch.optim.Adam(gcn.parameters(), lr=0.01, weight_decay=5e-4))

torch.manual_seed(42)
gat = GAT(dataset_cora.num_features, 8, dataset_cora.num_classes, heads=8, dropout=0.6)
gat_val, gat_test = run_model(gat, data_cora,
    torch.optim.Adam(gat.parameters(), lr=0.005, weight_decay=5e-4))

print('=== Cora Comparison ===')
print(f'{"Model":<8} | {"Val Acc":>8} | {"Test Acc":>9}')
print('-' * 32)
print(f'{"GCN":<8} | {gcn_val:>8.4f} | {gcn_test:>9.4f}')
print(f'{"GAT":<8} | {gat_val:>8.4f} | {gat_test:>9.4f}')

---
# MINI-PROJECT 4: Build 2-Head GAT and Analyze Attention

**Task:** Build a 2-head GAT on the Karate Club graph and deeply analyze what each attention head learns.

**Requirements:**
1. Implement a GAT with **2 attention heads** using `GATConv(heads=2)`
2. Train it on Karate Club for 300 epochs
3. Extract attention weights for **both heads separately**
4. Create a side-by-side visualization of what head 1 vs head 2 pays attention to
5. Find the top-5 highest-attention edges for each head and print them
6. In a markdown cell: do the two heads appear to learn different things?

**Bonus:** Also compare the node classification accuracy of 1-head vs 2-head vs 4-head GAT.

In [ ]:
# MINI-PROJECT SKELETON

class MultiHeadGAT(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=2, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        # TODO: define GATConv layers
        # First layer: heads=2, concat=True -> output dim = hidden_channels * heads
        # Last layer: heads=1, concat=False
        pass
    
    def forward(self, x, edge_index):
        # TODO
        pass
    
    def get_multi_head_attention(self, x, edge_index):
        """Return attention weights for ALL heads of layer 1."""
        self.eval()
        with torch.no_grad():
            # TODO: use return_attention_weights=True in conv1
            # Returns: (out, (edge_index, alpha)) where alpha shape is [E, heads]
            pass


torch.manual_seed(42)
model_multihead = MultiHeadGAT(
    in_channels=data_karate.num_features,
    hidden_channels=8,
    out_channels=dataset_karate.num_classes,
    heads=2
)

# TODO: train model_multihead on data_karate

# TODO: extract attention weights per head
# alpha shape: [num_edges, 2] — alpha[:, 0] = head 1, alpha[:, 1] = head 2

# TODO: create side-by-side visualization (head 1 vs head 2)

# TODO: find top-5 highest attention edges per head
# For head k: top_indices = alpha[:, k].argsort(descending=True)[:5]

print('Fill in the TODOs above!')

### Hints

<details>
<summary>Hint 1 — Getting per-head attention</summary>

```python
out, (edge_index_with_loops, alpha) = self.conv1(
    x, edge_index, return_attention_weights=True
)
# alpha shape: [num_edges_with_loops, num_heads]
head0_attention = alpha[:, 0]  # first head
head1_attention = alpha[:, 1]  # second head
```
</details>

<details>
<summary>Hint 2 — Top-k edges</summary>

```python
for head_idx in range(2):
    attn = alpha[:, head_idx]
    top_k = attn.argsort(descending=True)[:5]
    for idx in top_k:
        src = edge_index_with_loops[0, idx].item()
        dst = edge_index_with_loops[1, idx].item()
        print(f'Head {head_idx}: {src} -> {dst}, attention: {attn[idx]:.4f}')
```
</details>

---
## What's Next?

In **Notebook 5**, we tackle **inductive learning** — training GNNs that can generalize to *completely new graphs* at test time. This unlocks production use cases like predicting properties of new molecules or classifying new users in an evolving social network.